# Full Behavior Set Analysis

Extract and track non-template behavioral features across sessions:
- Rear detection and characterization (height, duration, rise time)
- Locomotion speed
- Hand-to-nose distance (grooming proxy)
- Feature distributions by session block (early/mid/late)
- Feature tracking over days by rat and task condition

In [ ]:
import sys
sys.path.insert(0, '.')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime
import pickle

from config import NODES, EDGES, NODE_IDX, DATA_ROOT
from data_io import load_session_df, load_sleap_dannce_keys, load_dannce_predictions
from skeleton import normalize_skeleton_batch
from processing import (
    get_rears, compute_speed, extract_session_features, find_template_matches,
)
from visualization import session_to_datetime, plot_feature_over_sessions

%matplotlib inline

In [ ]:
# Configuration
rear_threshold = 100  # z-height threshold for rear detection (calibration units)
fps_estimate = 50  # approximate SLEAP fps for speed calculations

In [ ]:
df = load_session_df()
print(f'Total sessions: {len(df)}')
df.head()

## Extract features from all sessions

In [ ]:
# Nested dict: summaries[rat][task][session] = features
summaries = {}

for i, row in df.iterrows():
    rat = row['rat']
    session = row['session']
    task = row['task']
    
    if rat not in summaries:
        summaries[rat] = {}
    if task not in summaries[rat]:
        summaries[rat][task] = {}
    if session in summaries[rat][task]:
        continue
    
    try:
        keys = load_sleap_dannce_keys(rat, session)
        dannce_3d = keys['dannce_keys_3D']
        if dannce_3d.ndim == 4:
            dannce_3d = dannce_3d.squeeze(axis=1).transpose(0, 2, 1)
        
        features = extract_session_features(
            dannce_3d, rear_thresh=rear_threshold, dt=1.0 / fps_estimate
        )
        
        # Block-based features (split session into 5-min blocks)
        block_frames = 5 * 60 * fps_estimate
        n_blocks = max(1, len(dannce_3d) // block_frames)
        
        rears_by_block = []
        speed_by_block = []
        for bi in range(n_blocks):
            start = bi * block_frames
            end = (bi + 1) * block_frames
            block_keys = dannce_3d[start:end]
            block_rears = get_rears(block_keys, start_thresh=rear_threshold)
            rears_by_block.append(len(block_rears))
            speed_by_block.append(np.mean(compute_speed(block_keys, dt=1.0 / fps_estimate)))
        
        features['rears_by_block'] = np.array(rears_by_block)
        features['speed_by_block'] = np.array(speed_by_block)
        
        summaries[rat][task][session] = features
        print(f'{rat}/{session} ({task}): {len(features["rears"])} rears')
        
    except Exception as e:
        print(f'{rat}/{session}: FAILED — {e}')

## Feature tracking over sessions

In [ ]:
def plot_all_features_over_sessions(summaries, rat, features_to_plot=None):
    """Plot multiple features over sessions for both task conditions."""
    if features_to_plot is None:
        features_to_plot = ['rear_height', 'rear_length', 'rise_time',
                            'LH_nose_dist', 'RH_nose_dist', 'nose_speed']
    
    n_features = len(features_to_plot)
    fig, axes = plt.subplots(n_features, 1, figsize=(14, 3 * n_features), sharex=True)
    if n_features == 1:
        axes = [axes]
    
    colors = {'lemon': 'orange', 'vanilla': 'purple'}
    
    for fi, feat in enumerate(features_to_plot):
        ax = axes[fi]
        
        for task in summaries.get(rat, {}):
            sessions = sorted(summaries[rat][task].keys())
            values = []
            valid_sessions = []
            
            for s in sessions:
                data = summaries[rat][task][s]
                if feat in data and len(data[feat]) > 0:
                    values.append(data[feat])
                    valid_sessions.append(s)
            
            if valid_sessions:
                plot_feature_over_sessions(
                    valid_sessions, values, feat,
                    ax=ax, color=colors.get(task, 'gray'), label=task
                )
        
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    
    plt.suptitle(f'{rat} — Feature evolution over sessions', fontsize=14)
    plt.tight_layout()
    plt.show()


for rat in sorted(summaries.keys()):
    plot_all_features_over_sessions(summaries, rat)

## Rear count by block over sessions

In [ ]:
def plot_rears_by_block_over_sessions(summaries, rat, task):
    """Heatmap of rear counts per 5-min block across sessions."""
    if task not in summaries.get(rat, {}):
        return
    
    sessions = sorted(summaries[rat][task].keys())
    valid = [(s, summaries[rat][task][s]) for s in sessions
             if 'rears_by_block' in summaries[rat][task][s]]
    
    if not valid:
        return
    
    max_blocks = max(len(v[1]['rears_by_block']) for v in valid)
    mat = np.full((len(valid), max_blocks), np.nan)
    labels = []
    
    for i, (s, data) in enumerate(valid):
        rb = data['rears_by_block']
        mat[i, :len(rb)] = rb
        labels.append(s)
    
    fig, ax = plt.subplots(figsize=(10, max(4, len(valid) * 0.35)))
    im = ax.imshow(mat, aspect='auto', cmap='YlOrRd', interpolation='nearest')
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel('5-min block')
    ax.set_title(f'{rat} ({task}) — Rears per 5-min block')
    plt.colorbar(im, label='Rear count')
    plt.tight_layout()
    plt.show()


for rat in sorted(summaries.keys()):
    for task in sorted(summaries[rat].keys()):
        plot_rears_by_block_over_sessions(summaries, rat, task)

## Speed by block over sessions

In [ ]:
def plot_speed_by_block(summaries, rat, task):
    """Heatmap of mean speed per block across sessions."""
    if task not in summaries.get(rat, {}):
        return
    
    sessions = sorted(summaries[rat][task].keys())
    valid = [(s, summaries[rat][task][s]) for s in sessions
             if 'speed_by_block' in summaries[rat][task][s]]
    
    if not valid:
        return
    
    max_blocks = max(len(v[1]['speed_by_block']) for v in valid)
    mat = np.full((len(valid), max_blocks), np.nan)
    labels = []
    
    for i, (s, data) in enumerate(valid):
        sb = data['speed_by_block']
        mat[i, :len(sb)] = sb
        labels.append(s)
    
    fig, ax = plt.subplots(figsize=(10, max(4, len(valid) * 0.35)))
    im = ax.imshow(mat, aspect='auto', cmap='viridis', interpolation='nearest')
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel('5-min block')
    ax.set_title(f'{rat} ({task}) — Mean speed per block')
    plt.colorbar(im, label='Speed (units/s)')
    plt.tight_layout()
    plt.show()


for rat in sorted(summaries.keys()):
    for task in sorted(summaries[rat].keys()):
        plot_speed_by_block(summaries, rat, task)

## Feature distribution comparison (task vs non-task)

In [ ]:
def plot_feature_distributions(summaries, rat, feature='rear_height', n_bins=30):
    """Compare feature distributions between task conditions."""
    fig, ax = plt.subplots(figsize=(8, 4))
    colors = {'lemon': 'orange', 'vanilla': 'purple'}
    
    for task in summaries.get(rat, {}):
        all_vals = []
        for s in summaries[rat][task]:
            data = summaries[rat][task][s]
            if feature in data and len(data[feature]) > 0:
                all_vals.extend(data[feature].tolist())
        
        if all_vals:
            ax.hist(all_vals, bins=n_bins, alpha=0.5,
                    color=colors.get(task, 'gray'), label=f'{task} (n={len(all_vals)})',
                    density=True)
    
    ax.set_xlabel(feature)
    ax.set_ylabel('Density')
    ax.set_title(f'{rat} — {feature} distribution by task')
    ax.legend()
    plt.tight_layout()
    plt.show()


for rat in sorted(summaries.keys()):
    for feat in ['rear_height', 'rear_length', 'LH_nose_dist', 'nose_speed']:
        plot_feature_distributions(summaries, rat, feature=feat)

## Save feature summaries

In [ ]:
with open('full_behavior_summaries.pkl', 'wb') as f:
    pickle.dump(summaries, f)

print('Saved to full_behavior_summaries.pkl')